# HiBASIL Tutorial 3: Finding the Epicenter of 1854 London Cholera Outbreak
This notebook introduces the BASIL (BAyesian Source Inference and Localization) framework. We will localize outbreak of 1854 London cholera using the historical concensus data, and compare to the real epicenter of the Broad Street pump that found out by John Snow!

## 1. Environment Setup
First, we prepare the workspace by creating directories and loading the necessary Bayesian engines.

In [1]:
import os
import numpy as np
import pandas as pd
# Set environment flags for PyTensor
os.environ['PYTENSOR_FLAGS'] = "base_compiledir=./pytensor_cache,cxx="

import pymc as pm
import arviz as az
from tqdm.auto import tqdm

import geopandas as gpd

import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

from sklearn.metrics import r2_score

# Create necessary directories automatically
for folder in ['./sim', './output', './pytensor_cache']:
    os.makedirs(folder, exist_ok=True)

import simulation  # simulation.py developed in this study
import hibasil       # core functions of the BASIL framework

np.random.seed(42)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
C:\Users\sunny\anaconda3\envs\mcenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ PyTensor cache: pytensor_cache


## 2. Historical Data
We will use the historical cholera concensus data (https://geodacenter.github.io/data-and-lab/snow/) and the Soho Street map (P. Li. cholera: Amend, Augment and Aid Analysis of John Snow’s Cholera
Map, 2017. URL https://cran.r-project.org/web/packages/cholera. R package version 0.9.1.)
We will use a single-focus model with Power-Law dispersal kernel to reconstruct the epidemic. Note the use of eps = 1e-12 to ensure numerical stability.

In [2]:
"""Main execution function for multi-row analysis"""        
obs_df = pd.read_csv("deaths_proportion.csv")
print("\nDataset summary:")
print(f"Total observations: {len(obs_df)}")

# plot the historical mortality records
# Load the Street Network
streets = gpd.read_file("./cholera_streets/cholera_streets_tm.shp")

sns.set_style(style='white')
fig, ax = plt.subplots(figsize=(7, 7))
merged_streets = streets.dissolve()
merged_streets.plot(ax=ax, color='grey', linewidth=0.7, alpha=0.5)

scatter = ax.scatter(obs_df['COORD_X'], obs_df['COORD_Y'], 
                         c=obs_df['death_proportion'], cmap='autumn_r', 
                         s=(obs_df['death_proportion'] + 0.1) * 100, 
                         edgecolors='black', linewidth=0.5,
                         label='Observed Severity')
    
cbar_scatter = fig.colorbar(scatter, ax=ax, pad=0.1, shrink=0.5)
cbar_scatter.set_label('Observed Severity', rotation=270, labelpad=15)

# label the Broad Street Pump
true_x0, true_y0 = 529396.5394, 181025.063 
ax.scatter(true_x0, true_y0, marker='*', color='black', s=150, label='Broad Street Pump')
plt.legend(loc='upper left')
plt.savefig(f"./output/Historical_cholera_observed_severity.png", bbox_inches='tight', dpi=300)


Dataset summary:
Total observations: 1852


## 3. The HiBASIL Model:Priors and Likelihood
We use weakly informed priors to guide the model without forcing a specific answer. Coordinates ($f_x$, $f_y$): Normal distribution centered on the area centroid. Intensity ($f_z$): Beta (1,1) for a flat unbiased start. Likelihood: Zero-Inflated Beta (ZOIB) to handle healthy and death simultaneously.

In [3]:
model_type = 'power_law'
model = hibasil.build_single_focus_model(obs_df, model_type=model_type)            

"sample from posterior"
with model:                
    print("\n--- Starting MCMC sampling ---")                    
    trace = pm.sample(draws=4000, tune=2000, chains=4, cores=2, target_accept=0.85, random_seed=42, nuts_sampler="nutpie")
    print("✓ Sampling completed!")

    # Generate Posterior Predictive Check (PPC)
    thin_trace = trace.sel(draw=slice(None, None, 5))  # use every 5th draw
    ppc = pm.sample_posterior_predictive(thin_trace, var_names=['obs'], progressbar=True, random_seed=42)

    # plot posterior predictive check
    y_sim = ppc.posterior_predictive['obs'].values                
    y_obs = obs_df['death_proportion'].values
    coord_xs = obs_df['COORD_X'].values
    coord_ys = obs_df['COORD_Y'].values
                
    hibasil.plot_posterior_predictive(y_sim, y_obs, coord_ys, model_type, sim='cholera')

# calculate posterior and ppc metrics
y_pred = ppc.posterior_predictive['obs'].mean(dim=['chain', 'draw']).values
y_low = ppc.posterior_predictive['obs'].quantile(0.025, dim=['chain', 'draw']).values
y_high = ppc.posterior_predictive['obs'].quantile(0.975, dim=['chain', 'draw']).values           
y_std = ppc.posterior_predictive['obs'].std(dim=['chain', 'draw']).values# Standard Deviation for each observation
        
posterior_df = pd.DataFrame({'coord_x':coord_xs, 'coord_y':coord_ys, 'y_obs':y_obs, 'y_pred':y_pred,
                                     'y_low':y_low, 'y_high':y_high, 'y_std':y_std})       
posterior_df.to_csv(f"./output/posterior_predictive_{model_type}.csv")

output_file = './output/ppc_metrics_all_sims.csv' 
pd.DataFrame(columns=['simulation', 'model_type', 'overall_r2', 'overall_rmse', 'overall_mae',
                        'obs_zeros', 'pred_zeros', 'obs_ones', 'pred_ones', 'continuous_r2', 
                        'continuous_rmse']).to_csv(output_file, index=False) # store all ppc metrics           
hibasil.compute_and_save_metrics(y_obs, y_pred, y_sim, model_type, 'cholera', output_file)


Multi-row model preparation:
- Total observations: 1852
- Exact zeros: 1483 (80.1%)
- Exact ones: 1 (0.1%)
- Continuous: 368 (19.9%)

--- Starting MCMC sampling ---


Progress,Draws,Divergences,Step Size,Gradients/Draw
,6000,0,0.35,7
,6000,0,0.34,15
,6000,0,0.38,7
,6000,0,0.35,15


✓ Sampling completed!


Sampling: [obs]


{'simulation': 'cholera',
 'model_type': 'power_law',
 'overall_r2': 0.09173362179615863,
 'overall_rmse': np.float64(0.0561492936854144),
 'overall_mae': 0.028316741674230265,
 'obs_zeros': np.float64(0.800755939524838),
 'pred_zeros': np.float64(0.8197462203023758),
 'obs_ones': np.float64(0.0005399568034557236),
 'pred_ones': np.float64(0.0009094897408207344),
 'continuous_r2': -0.9696138301775521,
 'continuous_rmse': np.float64(0.1106928892995419)}

## 4. Results and Spatial Visualization
We evaluate the model's accuracy by mapping the estimated focus against the true coordinates.

In [5]:
### 1. Check Convergence (R-hat should be <= 1.01)
summary = az.summary(trace, var_names=['fx1', 'fy1', 'fz1', 'scale1', 'exponent1'])
summary.to_csv(f"./output/model_{model_type}_trace_summary.csv")
print()
print(summary[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])
print()

### 2. Plot BASIL estimation results
"plot BASIL posterior prediction"
sns.set_style(style='white')
fig, ax = plt.subplots(figsize=(7, 7))
merged_streets = streets.dissolve()
merged_streets.plot(ax=ax, color='grey', linewidth=0.7, alpha=0.5)

"Create the interpolated surface from sampling locations"
# customize colormap
original_cmap = plt.cm.get_cmap('RdYlBu_r')
n_colors = 256
colors = original_cmap(np.linspace(0, 1, n_colors))
alpha_gradient = np.linspace(0, 1, int(n_colors * 0.3))
alphas = np.ones(n_colors)
alphas[:len(alpha_gradient)] = alpha_gradient
colors[:, 3] = alphas 
transparent_cmap = mcolors.ListedColormap(colors) # Create the new colormap

"creat contour surface"
cntr = ax.tricontourf(posterior_df['coord_x'], posterior_df['coord_y'], posterior_df['y_pred'], 
                      levels=30, cmap=transparent_cmap)

"Add 95% Credible Interval contour line for diagnostic uncertainty"
# sort all predicted values in descending order
sorted_preds = np.sort(posterior_df['y_pred'].values)[::-1]

# calculate the cumulative sum (the "volume")
cumulative_vol = np.cumsum(sorted_preds)
total_vol = cumulative_vol[-1]

# find the value threshold that captures 95% or 50% of that volume
threshold_95 = sorted_preds[np.argmax(cumulative_vol >= 0.95 * total_vol)]
threshold_50 = sorted_preds[np.argmax(cumulative_vol >= 0.50 * total_vol)]

#plot using these specific calculated thresholds
ax.tricontour(posterior_df['coord_x'], posterior_df['coord_y'], posterior_df['y_pred'], 
              levels=[threshold_95, threshold_50], colors='black', 
              linewidths=[0.5, 1.0], linestyles='--', # Thicker line for 50% core
              zorder=3, label=['95 % Credible Interval', '50 % Credible Interval'])

"Add the key localization markers"
# BASIL estimated location
est_x0 = summary.loc['fx1', 'mean']
est_y0 = summary.loc['fy1', 'mean']

ax.scatter(true_x0, true_y0, marker='*', color='black', s=150, label='Broad Street Pump')
ax.scatter(est_x0, est_y0, marker='o', facecolors='none', edgecolors='green', linewidth=1.5, label='BASIL Estimate', s=90)

custom_handles = [
    # The 95% Volume Isopleth (Thinner)
    Line2D([0], [0], color='black', lw=0.5, linestyle='--', label='95% Credible interval'),
    
    # The 50% Core Isopleth (Thicker)
    Line2D([0], [0], color='black', lw=1.0, linestyle='--', label='50% Core Risk'),
    
    # True Source (Solid Black Star - The Ground Truth)
    Line2D([0], [0], marker='*', color='w', markerfacecolor='black', 
           markeredgecolor='black', markersize=12, ls='', label='Broad Street Pump'),
        
    # BASIL Estimate (Green Circle)
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none', 
           markeredgecolor='green', markeredgewidth=1.5, markersize=10, ls='', label='Estimated Foci')
]

ax.legend(handles=custom_handles, loc='upper left', fontsize=9, frameon=True, numpoints=1)

plt.colorbar(cntr, label='Predicted Intensity', fraction=0.05, pad=0.04, shrink=0.6, format='%.1f')
plt.tight_layout()
plt.savefig("./output/BASIL_pred_cholera_severity.png", dpi=600)

print()
print("Congratulations! You have finished Tutorial 3!")


                 mean      hdi_3%     hdi_97%  r_hat
fx1        529426.124  529408.242  529443.400    1.0
fy1        181038.882  181023.132  181054.231    1.0
fz1             0.215       0.182       0.247    1.0
scale1        318.330     137.199     506.492    1.0
exponent1       1.966       1.144       2.821    1.0


Congratulations! You have finished Tutorial 3!
